# Week 4 · Class 2 — Two RAG demos

**Demo 1 — Constitution Capstone (PDF):** rebuild Class 1 pipeline → Gradio UI  
(chat · sources · k / threshold).

**Demo 2 — FIFA World Cup 2026 (web):** load Wikipedia with LangChain's **`WebBaseLoader`**,
index into in-memory Qdrant, answer **only** World Cup 2026 questions — then a second Gradio.

| | Demo 1 | Demo 2 |
|---|--------|--------|
| Source | Nepal Constitution PDF | Wikipedia page |
| Loader | `PyPDFLoader` | `WebBaseLoader` |
| Collection | `nepal_constitution` | `worldcup_2026` |
| UI | Gradio (port default) | Gradio (port 7861) |

## Before you start

- Groq key · constitution PDF for Demo 1 · **network** for Demo 2 (Wikipedia)

## Section 1 — Install packages

In [ ]:
!pip install -q langchain-core langchain-community langchain-text-splitters langchain-groq langchain-huggingface qdrant-client pypdf sentence-transformers gradio beautifulsoup4 lxml httpx

## Section 2 — Groq API key

In [ ]:
import os
import getpass

try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
except Exception:
    if not os.environ.get("GROQ_API_KEY"):
        os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your GROQ_API_KEY: ")

print("Key loaded:", bool(os.environ.get("GROQ_API_KEY")))

## Section 3 — Rebuild the RAG pipeline

Class 2 is **self-contained** — we rebuild the in-memory index here (Colab runtimes do not keep Class 1 state).

In [ ]:
from pathlib import Path
from langchain_groq import ChatGroq
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.2)

CANDIDATES = [
    Path("nepal-constitution.pdf"),
    Path("data/nepal-constitution.pdf"),
    Path("../data/nepal-constitution.pdf"),
    Path("/content/nepal-constitution.pdf"),
    Path("/content/data/nepal-constitution.pdf"),
]
PDF_PATH = next((p for p in CANDIDATES if p.exists()), None)
if PDF_PATH is None:
    raise FileNotFoundError("Upload nepal-constitution.pdf (see week-4/data/README.md)")

pages = PyPDFLoader(str(PDF_PATH)).load()
chunks = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200).split_documents(pages)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

COLLECTION = "nepal_constitution"
client = QdrantClient(":memory:")
client.create_collection(
    collection_name=COLLECTION,
    vectors_config=VectorParams(size=384, distance=Distance.COSINE),
)
texts = [c.page_content for c in chunks]
vectors = embeddings.embed_documents(texts)
client.upsert(
    collection_name=COLLECTION,
    points=[
        PointStruct(
            id=i,
            vector=vectors[i],
            payload={
                "text": chunks[i].page_content,
                "source": str(chunks[i].metadata.get("source", PDF_PATH.name)),
                "page": chunks[i].metadata.get("page"),
            },
        )
        for i in range(len(chunks))
    ],
)
print(f"Ready: {len(chunks)} chunks from {PDF_PATH.name}")

## Section 4 — `retrieve` + `ask` with tunable dials

In [ ]:
docs_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You answer questions about the Constitution of Nepal. "
     "Use ONLY the retrieved context below. Cite [page N] when available. "
     "If context is insufficient, say you do not find that in the constitution."),
    ("human", "Context:\n{context}\n\nQuestion: {question}"),
])

def retrieve(question: str, k: int = 4, score_threshold: float = 0.25):
    hits = client.query_points(
        collection_name=COLLECTION,
        query=embeddings.embed_query(question),
        limit=int(k),
    ).points
    return [
        {
            "text": h.payload["text"],
            "source": h.payload.get("source"),
            "page": h.payload.get("page"),
            "score": float(h.score),
        }
        for h in hits
        if h.score >= float(score_threshold)
    ]

def format_hits(hits):
    blocks = []
    for h in hits:
        tag = f"page {h['page']}" if h.get("page") is not None else h.get("source", "doc")
        blocks.append(f"**[{tag} | score={h['score']:.3f}]**\n{h['text']}")
    return "\n\n".join(blocks) if blocks else "_No chunks above the score threshold._"

def ask(question: str, k: int = 4, score_threshold: float = 0.25):
    hits = retrieve(question, k=k, score_threshold=score_threshold)
    if not hits:
        answer = "I could not find that in the constitution (no chunk passed the score threshold)."
    else:
        answer = (docs_prompt | llm).invoke({
            "question": question,
            "context": format_hits(hits),
        }).content
    return answer, hits

answer, hits = ask("What rights relate to equality?")
print(answer)
print("hits:", len(hits))

## Section 5 — Gradio Blocks layout

Layout goals:

1. **Chat** — user questions + grounded answers
2. **Sources** — markdown snippets with page + score
3. **Controls** — sliders for `k` and score threshold

In [ ]:
import gradio as gr

def respond(message, history, k, score_threshold):
    answer, hits = ask(message, k=k, score_threshold=score_threshold)
    sources_md = format_hits(hits)
    # Gradio Chatbot messages format (role/content dicts)
    history = history or []
    history = list(history) + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": answer},
    ]
    return history, sources_md

with gr.Blocks(title="Nepal Constitution RAG") as demo:
    gr.Markdown("# Nepal Constitution RAG\nGrounded answers from an in-memory Qdrant index.")
    with gr.Row():
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(height=420, label="Chat")
            msg = gr.Textbox(label="Ask the constitution", placeholder="e.g. What does it say about equality?")
            with gr.Row():
                k = gr.Slider(1, 8, value=4, step=1, label="top-k")
                threshold = gr.Slider(0.0, 0.8, value=0.25, step=0.05, label="score threshold")
            clear = gr.Button("Clear")
        with gr.Column(scale=2):
            sources = gr.Markdown("_Retrieved sources will appear here._", label="Sources")

    msg.submit(respond, [msg, chatbot, k, threshold], [chatbot, sources]).then(
        lambda: "", None, msg
    )
    clear.click(lambda: ([], "_Retrieved sources will appear here._"), None, [chatbot, sources])

demo.launch(share=False)

### Tips (Demo 1 — Constitution Gradio)

- In Colab, `demo.launch()` prints a local URL; set `share=True` for a temporary public link
- Stop or interrupt the Gradio cell before starting **Demo 2** (or use a different port)
- Capstone deliverable is still this Constitution UI

## Demo 1 checklist

- [ ] Gradio chat grounded in constitution chunks
- [ ] Sources panel + k / threshold sliders work

---

# Demo 2 — FIFA World Cup 2026 bot (Wikipedia + `WebBaseLoader`)

**Work together in class.** Same RAG spine as Demo 1 — only the **document loader** and
**system prompt** change.

**Web source:** https://en.wikipedia.org/wiki/2026_FIFA_World_Cup

`WebBaseLoader` fetches that page and turns it into LangChain `Document`s (this is the
web-source loader from the Class 1 loader menu).

## Section 7 — Load Wikipedia with `WebBaseLoader`

No PDF upload. The data comes from the live web page.

In [ ]:
from langchain_community.document_loaders import WebBaseLoader

WC_URL = "https://en.wikipedia.org/wiki/2026_FIFA_World_Cup"

# Wikipedia may block bare scripts — send a simple User-Agent.
wc_loader = WebBaseLoader(
    web_paths=[WC_URL],
    header_template={"User-Agent": "LFConnectBootcamp/1.0 (classroom RAG demo)"},
)
wc_docs = wc_loader.load()

print(f"Loaded {len(wc_docs)} document(s) via WebBaseLoader")
print("source:", wc_docs[0].metadata.get("source"))
print("title:", wc_docs[0].metadata.get("title"))
print("--- preview (first 500 chars) ---")
print(wc_docs[0].page_content[:500])

## Section 8 — Chunk + store in collection `worldcup_2026`

Reuse MiniLM embeddings from Demo 1. Keep a **separate** Qdrant collection so the
constitution index stays intact.

In [ ]:
WC_COLLECTION = "worldcup_2026"

wc_chunks = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
).split_documents(wc_docs)
print(f"World Cup chunks: {len(wc_chunks)}")

try:
    client.delete_collection(WC_COLLECTION)
except Exception:
    pass

client.create_collection(
    collection_name=WC_COLLECTION,
    vectors_config=VectorParams(size=384, distance=Distance.COSINE),
)

wc_texts = [c.page_content for c in wc_chunks]
wc_vectors = embeddings.embed_documents(wc_texts)
client.upsert(
    collection_name=WC_COLLECTION,
    points=[
        PointStruct(
            id=i,
            vector=wc_vectors[i],
            payload={
                "text": wc_chunks[i].page_content,
                "source": WC_URL,
                "title": wc_chunks[i].metadata.get("title", "2026 FIFA World Cup"),
            },
        )
        for i in range(len(wc_chunks))
    ],
)
print("Points in worldcup_2026:", client.count(WC_COLLECTION).count)

## Section 9 — Ask the World Cup bot (domain-locked)

Grounded on Wikipedia retrieval. Refuses off-topic questions (constitution, other sports, etc.).

In [ ]:
wc_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a 2026 FIFA World Cup assistant. "
     "Answer ONLY using the retrieved Wikipedia context below. "
     "Stay on the 2026 FIFA World Cup (hosts, venues, format, teams, facts in the context). "
     "If the question is unrelated, say you only answer questions about the 2026 FIFA World Cup. "
     "If the context lacks the answer, say you do not find that on the Wikipedia page."),
    ("human", "Context:\n{context}\n\nQuestion: {question}"),
])

def retrieve_worldcup(question: str, k: int = 4, score_threshold: float = 0.25):
    hits = client.query_points(
        collection_name=WC_COLLECTION,
        query=embeddings.embed_query(question),
        limit=int(k),
    ).points
    return [
        {
            "text": h.payload["text"],
            "source": h.payload.get("source"),
            "title": h.payload.get("title"),
            "score": float(h.score),
        }
        for h in hits
        if h.score >= float(score_threshold)
    ]

def format_wc_hits(hits):
    if not hits:
        return "_No chunks above the score threshold._"
    return "\n\n".join(
        f"**[score={h['score']:.3f} | Wikipedia]**\n{h['text'][:800]}" for h in hits
    )

def ask_worldcup(question: str, k: int = 4, score_threshold: float = 0.25):
    hits = retrieve_worldcup(question, k=k, score_threshold=score_threshold)
    if not hits:
        return (
            "I could not find that on the 2026 World Cup Wikipedia page "
            "(no chunk passed the score threshold).",
            hits,
        )
    answer = (wc_prompt | llm).invoke({
        "question": question,
        "context": format_wc_hits(hits),
    }).content
    return answer, hits

# Class smoke tests
for q in [
    "Which countries are hosting the 2026 FIFA World Cup?",
    "How many teams will play in 2026?",
    "What does the Nepal constitution say about equality?",  # should stay on World Cup lane
]:
    print("Q:", q)
    ans, hits = ask_worldcup(q)
    print("A:", ans[:450])
    print("hits:", len(hits), "\n---")

### Work-together questions

Students call these out; run `ask_worldcup` live:

1. Who hosts in 2026?  
2. What changed in the tournament format?  
3. Name a host city or venue from the page  
4. Something off-topic — confirm refusal  

In [ ]:
# Live class question
q = "Which countries are hosting the 2026 World Cup?"
answer, hits = ask_worldcup(q)
print(answer)
print()
print(format_wc_hits(hits))

## Section 10 — Demo 2 Gradio UI (Wikipedia World Cup)

Same Blocks layout as Demo 1: chat · sources · k · threshold.  
Runs on **port 7861** so it can sit beside the constitution app.

In [ ]:
def respond_wc(message, history, k, score_threshold):
    answer, hits = ask_worldcup(message, k=k, score_threshold=score_threshold)
    history = list(history or []) + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": answer},
    ]
    return history, format_wc_hits(hits)

with gr.Blocks(title="2026 World Cup RAG") as wc_demo:
    gr.Markdown(
        "# 2026 FIFA World Cup Bot\n"
        "Data: Wikipedia via `WebBaseLoader` · in-memory Qdrant · Groq"
    )
    with gr.Row():
        with gr.Column(scale=3):
            wc_chat = gr.Chatbot(height=420, label="Chat")
            wc_msg = gr.Textbox(
                label="Ask about World Cup 2026",
                placeholder="e.g. Which countries are hosting?",
            )
            with gr.Row():
                wc_k = gr.Slider(1, 8, value=4, step=1, label="top-k")
                wc_thr = gr.Slider(0.0, 0.8, value=0.25, step=0.05, label="score threshold")
            wc_clear = gr.Button("Clear")
        with gr.Column(scale=2):
            wc_sources = gr.Markdown("_Retrieved Wikipedia chunks will appear here._")

    wc_msg.submit(
        respond_wc, [wc_msg, wc_chat, wc_k, wc_thr], [wc_chat, wc_sources]
    ).then(lambda: "", None, wc_msg)
    wc_clear.click(
        lambda: ([], "_Retrieved Wikipedia chunks will appear here._"),
        None,
        [wc_chat, wc_sources],
    )

wc_demo.launch(share=False, server_port=7861)

## Demo 2 takeaway

| Demo 1 | Demo 2 |
|--------|--------|
| `PyPDFLoader` → constitution PDF | `WebBaseLoader` → Wikipedia URL |
| Legal grounded prompt | Football grounded prompt |
| Same chunk → MiniLM → Qdrant → Groq → Gradio | Same |

**Specializing a bot = swap the loader + tighten the system prompt.** The RAG spine stays.